# Datacamp Intermediate Python: Ch4 Multi-Input & Multi-Output models

In [24]:
from startup import np, pd, Path, smf, plt, sns
from zipfile import ZipFile
import re
from PIL import Image

In [2]:
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.nn.init as init

In [3]:
from torchmetrics import Accuracy, Precision, Recall, F1Score, MeanSquaredError
# For CNN
from torchvision.datasets import ImageFolder
from torchvision import transforms

In [4]:
from ydf import RandomForestLearner

In [36]:
data_dir = Path.home() / 'Work' / 'Data' / 'datacamp_intermediate_pytorch'
data_file_train = data_dir / 'omniglot_train.zip'
data_file_test = data_dir / 'omniglot_test.zip'
zip_file_train = ZipFile(data_file_train)
zip_file_test = ZipFile(data_file_test)

In [15]:
example_filename = 'omniglot_train/Gujarati/character42/0459_14.png'
def extract_metadata(filename):
    return re.match(r'omniglot_(train|test)/([^/]+)/([^/]+)/(\w+\.*png)', filename)
extract_metadata(example_filename)

<re.Match object; span=(0, 47), match='omniglot_train/Gujarati/character42/0459_14.png'>

In [43]:
def create_samples(zip_file, ohe=True):
    dict_files = {
        fn: md.groups() for fn, md in
        [(f.filename, extract_metadata(f.filename)) for f in zip_file.infolist()] if md is not None
    }

    df_files = pd.DataFrame.from_dict(
        dict_files,
        orient='index',
        columns=['set', 'alphabet', 'character', 'filename']
    ).assign(
        alphabet_character = lambda df: df.alphabet + '_' + df.character,
    )
    alphabets = sorted(df_files.alphabet.unique())
    alphabet_chars = sorted(df_files.alphabet_character.unique())
    df_meta = df_files.assign(
        alphabet_idx = lambda df: df.alphabet.map(alphabets.index),
        alphabet_char_idx = lambda df: df.alphabet_character.map(alphabet_chars.index)
    )
    # Create samples of (filename, OHE alphabet, character index)
    n_alphabets = len(alphabets)
    return list(zip(df_meta.index.values, np.eye(n_alphabets)[df_meta.alphabet_idx] if ohe else df_meta.alphabet_idx.astype(int), df_meta.alphabet_char_idx.astype(int)))

In [44]:
samples_train = create_samples(zip_file_train)
samples_test = create_samples(zip_file_test)

Dataset from course - need to modify to use df_train_files or generate expected samples data structure

In [90]:
class OmniglotDataset(Dataset):
    def __init__(self, transform, samples):
		# Assign transform and samples to class attributes
        self.transform = transform
        self.samples = samples

    def __len__(self):
		# Return number of samples
        return len(self.samples)

    def __getitem__(self, idx):
      	# Unpack the sample at index idx
        img_path, alphabet, label = self.samples[idx]
        # Original: img = Image.open(img_path).convert('L')
        img = Image.open(
            ZipFile(
                (data_dir / img_path.split('/')[0]).as_posix() + '.zip'
            ).open(img_path)).convert('L')
        # Transform the image
        img_transformed = self.transform(img)
        return img_transformed, torch.tensor(alphabet, dtype=torch.float32), label

Model from course

In [53]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        # Define sub-networks as sequential models
        self.image_layer = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.MaxPool2d(kernel_size=2),
            nn.ELU(),
            nn.Flatten(),
            nn.Linear(16*32*32, 128)
        )
        self.alphabet_layer = nn.Sequential(
            nn.Linear(30, 8),
            nn.ELU(),
        )
        self.classifier = nn.Sequential(
            nn.Linear(128 + 8, 964),
        )

    def forward(self, x_image, x_alphabet):
		# Pass the x_image and x_alphabet through appropriate layers
        x_image = self.image_layer(x_image)
        x_alphabet = self.alphabet_layer(x_alphabet)
        # Concatenate x_image and x_alphabet
        x = torch.cat((x_image, x_alphabet), dim=1)
        return self.classifier(x)

In [91]:
dataset_train = OmniglotDataset(
    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Resize((64,64))
    ]),
    samples=samples_train
)

In [97]:
dataloader_train = DataLoader(
    dataset_train, batch_size=3, shuffle=True
)

In [98]:
net = Net()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.01)


In [99]:
# Takes 1h 36min for 10 epochs with batch size 3
# - Faster if use unzipped images?
def train(net, criterion, optimizer, dataloader, epochs=10):
    for epoch in range(epochs):
        running_loss = 0.0
        for img, alpha, label in dataloader:
            optimizer.zero_grad()
            output = net(img, alpha)
            loss = criterion(output, label)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        epoch_loss = running_loss / len(dataloader)
        print(f"Epoch {epoch+1}, Loss: {epoch_loss:.4f}")

Epoch 1, Loss: 6.5792
Epoch 2, Loss: 4.3800
Epoch 3, Loss: 3.1544
Epoch 4, Loss: 2.4496
Epoch 5, Loss: 1.7657
Epoch 6, Loss: 1.1292
Epoch 7, Loss: 0.6508
Epoch 8, Loss: 0.3468
Epoch 9, Loss: 0.1772
Epoch 10, Loss: 0.0986


In [103]:
model_file = data_dir / 'omniglot_net.pth'
if not model_file.exists():
    train(net, criterion, optimizer, dataloader_train, epochs=10)
    torch.save(net.state_dict(), model_file)
else:
    net.load_state_dict(torch.load(model_file))

True

In [100]:
dataset_test = OmniglotDataset(
    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Resize((64,64))
    ]),
    samples=samples_test
)

dataloader_test = DataLoader(
    dataset_test, batch_size=3, shuffle=True
)


In [101]:
# Evaluate the model on the test set:
# takes 7min for 12.1k images and gives accuracy 0.7191
acc = Accuracy(task='multiclass', num_classes=964)

net.eval()
with torch.no_grad():
    for img, alpha, label in dataloader_test:
        output = net(img, alpha)
        acc.update(preds=output.argmax(dim=1), target=label)
print(f"Test Accuracy: {acc.compute():.4f}")

Test Accuracy: 0.7191


Multi-Output Model


In [ ]:
samples = create_samples(zip_file_train, ohe=False)

# Print the sample at index 100
print(samples[100])

# Create dataset_train
dataset_train = OmniglotDataset(
    transform=transforms.Compose([
        transforms.ToTensor(),
      	transforms.Resize((64, 64)),
    ]),
    samples=samples,
)

# Create dataloader_train
dataloader_train = DataLoader(
    dataset_train, shuffle=True, batch_size=32,
)

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.image_layer = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.MaxPool2d(kernel_size=2),
            nn.ELU(),
            nn.Flatten(),
            nn.Linear(16*32*32, 128)
        )
        # Define the two classifier layers
        self.classifier_alpha = nn.Linear(128, 30)
        self.classifier_char = nn.Linear(128, 964)

    def forward(self, x):
        x_image = self.image_layer(x)
        # Pass x_image through the classifiers and return both results
        output_alpha = self.classifier_alpha(x_image)
        output_char = self.classifier_char(x_image)
        return output_alpha, output_char

In [ ]:
net = Net()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.05)

for epoch in range(1):
    for images, labels_alpha, labels_char in dataloader_train:
        optimizer.zero_grad()
        outputs_alpha, outputs_char = net(images)
        # Compute alphabet classification loss
        loss_alpha = criterion(outputs_alpha, labels_alpha)
        # Compute character classification loss
        loss_char = criterion(outputs_char, labels_char)
        # Compute total loss
        loss = loss_alpha + loss_char
        loss.backward()
        optimizer.step()